# Notebook 06 — Chunker Bake-Off (T182)

Compares three chunker variants against the 15-triple RAG golden set.
Reports `hit@5`, `MRR`, and mean retrieval latency per variant.
Emits a Markdown comparison table and records the winner in `docs/DECISIONS.md`.

**Prerequisites**: compose stack running, `EMBEDDING_API_KEY` set in env.

For the automated version that runs in CI, see
`backend/tests/evals/rag/test_chunker_comparison.py`.

In [ ]:
import subprocess, sys
# Install deps if running outside the uv env
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '../backend', '-q'], check=True)

In [ ]:
import json, os, pathlib, statistics, time, uuid
import asyncio

GOLDEN = pathlib.Path('../backend/tests/evals/rag/golden.jsonl')
THRESHOLDS = pathlib.Path('../eval_thresholds.yaml')
DECISIONS_MD = pathlib.Path('../docs/DECISIONS.md')

rows = [json.loads(l) for l in GOLDEN.read_text().splitlines() if l.strip()]
print(f'Loaded {len(rows)} golden rows')

In [ ]:
# Import page bodies from the test module so we use the same seed corpus
sys.path.insert(0, '../backend')
from tests.evals.rag.test_rag_quality import _PAGE_BODIES
print(f'Seed pages: {len(_PAGE_BODIES)}')

In [ ]:
from app.frameworks.config import get_settings
settings = get_settings()
assert settings.embedding_api_key, 'Set EMBEDDING_API_KEY before running this notebook'
print(f'Embedding provider: {settings.embedding_provider}')

In [ ]:
import importlib
from sqlalchemy import text
from sqlalchemy.ext.asyncio import create_async_engine, AsyncSession
from sqlalchemy.pool import NullPool
from app.adapters.embeddings.hosted_embeddings import HostedEmbeddings
from app.adapters.repositories.chunk_repository import PostgresChunkRepository
from app.use_cases.rag_search import RAGSearchUseCase
import app.use_cases.reindex_tenant_chunks as rtc

engine = create_async_engine(settings.migration_database_url, poolclass=NullPool)

async def run_variant(chunker_name):
    os.environ['CHUNKER'] = chunker_name
    importlib.reload(rtc)
    tenant_id = uuid.uuid4()
    widget_id = uuid.uuid4()
    page_ids = {}

    async with engine.begin() as conn:
        await conn.execute(text('INSERT INTO tenants (id, slug, display_name) VALUES (:id, :slug, :dn)'),
            {'id': str(tenant_id), 'slug': f'nb-bakeoff-{chunker_name}-{tenant_id.hex[:6]}', 'dn': f'NB Bakeoff {chunker_name}'})
        await conn.execute(text('INSERT INTO widgets (id, tenant_id, public_id) VALUES (:id, :tid, :pub)'),
            {'id': str(widget_id), 'tid': str(tenant_id), 'pub': f'pub-{widget_id.hex[:8]}'})
        for page_key, body in _PAGE_BODIES.items():
            pid = uuid.uuid4()
            page_ids[page_key] = pid
            await conn.execute(text(
                'INSERT INTO cms_pages (id, tenant_id, title, body, state, slug) '
                'VALUES (:id, :tid, :title, :body, \'published\', :slug)'),
                {'id': str(pid), 'tid': str(tenant_id), 'title': f'Page {page_key[-4:]}',
                 'body': body, 'slug': f'nb-page-{page_key[-4:]}-{chunker_name}'})

    embedder = HostedEmbeddings(provider=settings.embedding_provider,
                                api_key=settings.embedding_api_key,
                                model=settings.embedding_model)
    async with AsyncSession(engine) as db:
        chunk_repo = PostgresChunkRepository(db)
        reindex = rtc.ReindexTenantChunksUseCase(chunk_repo, embedder)
        for page_key, body in _PAGE_BODIES.items():
            await reindex.execute(cms_page_id=page_ids[page_key], tenant_id=tenant_id, body=body)
        await db.commit()

    hits, rrs, latencies = 0, [], []
    async with AsyncSession(engine) as db:
        rag = RAGSearchUseCase(PostgresChunkRepository(db), embedder)
        for row in rows:
            expected_pid = page_ids.get(row['expected_cms_page_id'])
            if not expected_pid: continue
            t0 = time.perf_counter()
            result = await rag.execute(query=row['query'], tenant_id=tenant_id)
            latencies.append((time.perf_counter() - t0) * 1000)
            retrieved = [str(c.cms_page_id) for c in result.chunks]
            if str(expected_pid) in retrieved:
                hits += 1
                rrs.append(1.0 / (retrieved.index(str(expected_pid)) + 1))
            else:
                rrs.append(0.0)

    async with engine.begin() as conn:
        await conn.execute(text('DELETE FROM tenants WHERE id = :id'), {'id': str(tenant_id)})

    n = len(rows)
    p95 = sorted(latencies)[int(0.95 * len(latencies))] if latencies else 0
    return dict(chunker=chunker_name, hit_at_5=hits/n, mrr=sum(rrs)/n,
                mean_ms=statistics.mean(latencies) if latencies else 0, p95_ms=p95)

print('Functions defined.')

In [ ]:
results = []
for name in ['fixed_500', 'paragraph_aware', 'header_first']:
    print(f'Running {name}...')
    r = await run_variant(name)
    results.append(r)
    print(f'  hit@5={r["hit_at_5"]:.3f}  MRR={r["mrr"]:.3f}  mean={r["mean_ms"]:.0f}ms  p95={r["p95_ms"]:.0f}ms')

In [ ]:
import yaml
thresholds = yaml.safe_load(THRESHOLDS.read_text())
recall_threshold = thresholds['rag_golden_set_recall_at_5']

eligible = [r for r in results if r['hit_at_5'] >= recall_threshold and r['p95_ms'] <= 200]
winner = sorted(eligible, key=lambda r: (-r['hit_at_5'], -r['mrr'], r['mean_ms']))[0] if eligible else None

print('\n| chunker | hit@5 | MRR | mean_ms | p95_ms | eligible |')
print('|---------|-------|-----|---------|--------|----------|')
for r in results:
    e = '✓' if r['hit_at_5'] >= recall_threshold and r['p95_ms'] <= 200 else '✗'
    mark = ' ← WINNER' if winner and r['chunker'] == winner['chunker'] else ''
    print(f"| {r['chunker']}{mark} | {r['hit_at_5']:.3f} | {r['mrr']:.3f} | {r['mean_ms']:.0f} | {r['p95_ms']:.0f} | {e} |")

print(f'\nWinner: {winner["chunker"] if winner else "none"}')